# Using Lysimeter Data in Drought-Response Experiments to Predict Performance Across Camelina Genotypes​
## William B, Nibir N, and Brianne S
### CMPT/PLSC 878​
## Dataset Visualization Script
### ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Import statements:

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import helpers

# Directory/File Name settings

In [ ]:
dataset_directory = "CamelinaMAGIC-Dataset-Complete/"

MAGIC1_daily_transpiration_file = os.path.join(dataset_directory, "CamelinaMAGIC1.0-DailyTranspiration.csv")
MAGIC1_weight_raw = os.path.join(dataset_directory, "CamelinaMAGIC1.0-WeightRaw.csv")
MAGIC1_weather_station = os.path.join(dataset_directory, "CamelinaMAGIC1.0-WeatherStation.csv")

MAGIC2_daily_transpiration_file = os.path.join(dataset_directory, "CamelinaMAGIC2.0-DailyTranspiration.csv")
MAGIC2_weight_raw = os.path.join(dataset_directory, "CamelinaMAGIC2.0-WeightRaw.csv")
MAGIC2_weather_station = os.path.join(dataset_directory, "CamelinaMAGIC2.0-WeatherStation.csv")

MAGIC3_daily_transpiration_file = os.path.join(dataset_directory, "CamelinaMAGIC3.0-DailyTranspiration.csv")
MAGIC3_weight_raw = os.path.join(dataset_directory, "CamelinaMAGIC3.0-WeightRaw.csv")
MAGIC3_weather_station = os.path.join(dataset_directory, "CamelinaMAGIC3.0-WeatherStation.csv")

MAGIC_randomization_file = os.path.join(dataset_directory, "CamelinaMAGIC-ALL-Randomization.xlsx")

output_directory = "output/"

# Dataset Visualization Code:

In [ ]:
# Load Experiment 1 data
weight_df_1 = helpers.read_weight_csv(MAGIC1_weight_raw, MAGIC_randomization_file, 1)

# Plot Experiment 1
helpers.plot_drought_weight_df(weight_df_1)
helpers.plot_control_weight_df(weight_df_1)

# Load Experiment 2 data
weight_df_2 = helpers.read_weight_csv(MAGIC2_weight_raw, MAGIC_randomization_file, 2)

# Plot Experiment 2
helpers.plot_drought_weight_df(weight_df_2)
helpers.plot_control_weight_df(weight_df_2)

# Load Experiment 3 data
weight_df_3 = helpers.read_weight_csv(MAGIC3_weight_raw, MAGIC_randomization_file, 3)

# Plot Experiment 3
helpers.plot_drought_weight_df(weight_df_3)
helpers.plot_control_weight_df(weight_df_3)

# Transpiration Data Visualization

In [ ]:
# Load Experiment 1 transpiration data
transpiration_df_1 = helpers.read_transpiration_csv(MAGIC1_daily_transpiration_file, MAGIC_randomization_file, 1)

# Plot Experiment 1 transpiration
helpers.plot_drought_transpiration_df(transpiration_df_1)
helpers.plot_control_transpiration_df(transpiration_df_1)

# Load Experiment 2 transpiration data
transpiration_df_2 = helpers.read_transpiration_csv(MAGIC2_daily_transpiration_file, MAGIC_randomization_file, 2)

# Plot Experiment 2 transpiration
helpers.plot_drought_transpiration_df(transpiration_df_2)
helpers.plot_control_transpiration_df(transpiration_df_2)

# Load Experiment 3 transpiration data
transpiration_df_3 = helpers.read_transpiration_csv(MAGIC3_daily_transpiration_file, MAGIC_randomization_file, 3)

# Plot Experiment 3 transpiration
helpers.plot_drought_transpiration_df(transpiration_df_3)
helpers.plot_control_transpiration_df(transpiration_df_3)

# Weather Station Data Visualization

In [ ]:
# Load Experiment 1 weather station data
weather_df_1 = helpers.read_weather_station_csv(MAGIC1_weather_station, 1)

# Load Experiment 2 weather station data
weather_df_2 = helpers.read_weather_station_csv(MAGIC2_weather_station, 2)

# Load Experiment 3 weather station data
weather_df_3 = helpers.read_weather_station_csv(MAGIC3_weather_station, 3)

# Enhanced weather station visualization with daily min/max/mean statistics
helpers.plot_weather_station_data_enhanced({
    "Experiment 1": weather_df_1,
    "Experiment 2": weather_df_2,
    "Experiment 3": weather_df_3
}, show_debug=False)

# Genotype-Specific Analysis: Control vs Drought Comparison

In [ ]:
# Select a genotype to analyze
genotype_to_analyze = 'TMP23992'

# Plot weight comparison for the selected genotype
# helpers.plot_genotype_weight_df(weight_df_3, genotype_to_analyze)

# Plot transpiration comparison for the selected genotype
helpers.plot_genotype_transpiration_df(transpiration_df_3, genotype_to_analyze)

genotype_to_analyze = 'CAM236'

# Plot weight comparison for the selected genotype (Experiment 1)
# helpers.plot_genotype_weight_df(weight_df_1, genotype_to_analyze)

# Plot transpiration comparison for the selected genotype (Experiment 1)
helpers.plot_genotype_transpiration_df(transpiration_df_1, genotype_to_analyze)

# Sample-Specific Transpiration Analysis

In [ ]:
# Prepare ALL drought sample data with weight information (not just selected samples)
# Combine all data
all_transpiration = pd.concat([transpiration_df_1, transpiration_df_2, transpiration_df_3])
all_weight = pd.concat([weight_df_1, weight_df_2, weight_df_3])

# Filter for drought samples only
drought_transpiration = all_transpiration[all_transpiration['Treatment'] == 'DR_100'].copy()
drought_weight = all_weight[all_weight['Treatment'] == 'DR_100'].copy()

# Get unique drought samples
unique_samples = drought_transpiration[['Sample', 'Genotype', 'Experiment']].drop_duplicates()

print(f"Total drought samples: {len(unique_samples)}")
print(f"\nSamples by genotype:")
print(unique_samples.groupby('Genotype').size())

# Prepare sample data for all drought samples
all_drought_sample_data = {}

for _, row in unique_samples.iterrows():
    sample_name = row['Sample']
    genotype = row['Genotype']
    experiment_num = row['Experiment']
    
    # Filter by sample name, genotype, AND experiment number
    trans = drought_transpiration[
        (drought_transpiration['Sample'] == sample_name) & 
        (drought_transpiration['Genotype'] == genotype) &
        (drought_transpiration['Experiment'] == experiment_num)
    ].sort_values('Days').copy()
    
    weight = drought_weight[
        (drought_weight['Sample'] == sample_name) & 
        (drought_weight['Genotype'] == genotype) &
        (drought_weight['Experiment'] == experiment_num)
    ].sort_values('Days')[['Days', 'Sample', 'Weight']]
    
    # Skip if insufficient data
    if len(trans) < 2 or len(weight) < 2:
        continue
    
    # Merge on Days and Sample
    merged = pd.merge(trans, weight, on=['Days', 'Sample'], how='left')
    
    # Calculate transpiration as % of weight
    merged['Transpiration_Pct'] = (merged['Transpiration'] / merged['Weight']) * 100
    
    # Create unique key for each sample
    sample_key = f"{genotype}_{sample_name}_Exp{experiment_num}"
    all_drought_sample_data[sample_key] = {
        'genotype': genotype,
        'sample': sample_name,
        'experiment': experiment_num,
        'data': merged
    }

print(f"\nSuccessfully loaded {len(all_drought_sample_data)} drought samples with complete data")

In [ ]:
# Calculate drought scores for ALL drought samples
periods = [
    ('s1 (Days 1-20)', 1, 20),
    ('s2 (Days 21-25)', 21, 25),
    ('s3 (Days 36-40)', 36, 40)
]

print("=" * 80)
print("CALCULATING DROUGHT SCORES FOR ALL DROUGHT SAMPLES")
print("=" * 80)

# Drought score weights
w1 = 1
w2 = 1

# Store individual sample drought scores
sample_drought_scores = []

for sample_key, sample_info in all_drought_sample_data.items():
    genotype = sample_info['genotype']
    sample_name = sample_info['sample']
    experiment = sample_info['experiment']
    data = sample_info['data']
    
    # Calculate period statistics
    results = helpers.analyze_sample_periods(data, sample_name, periods, use_percent=True)
    
    # Extract s1, s2, s3 values
    s1 = results['periods']['s1 (Days 1-20)']['mean_daily_change']
    s2 = results['periods']['s2 (Days 21-25)']['mean_daily_change']
    s3 = results['periods']['s3 (Days 36-40)']['mean_daily_change']
    
    # Skip if any value is None
    if s1 is None or s2 is None or s3 is None:
        continue
    
    # Calculate drought score components
    component1 = -s2 / s1
    component2 = min(1, s3 / s1)
    drought_score = w1 * component1 + w2 * component2
    
    # Store results
    sample_drought_scores.append({
        'Sample_Key': sample_key,
        'Genotype': genotype,
        'Sample': sample_name,
        'Experiment': experiment,
        's1': s1,
        's2': s2,
        's3': s3,
        'Component1': component1,
        'Component2': component2,
        'Drought_Score': drought_score
    })

# Create DataFrame
drought_scores_df = pd.DataFrame(sample_drought_scores)

# Calculate mean and std for each genotype
genotype_stats = drought_scores_df.groupby('Genotype')['Drought_Score'].agg(['mean', 'std', 'count'])
genotype_stats = genotype_stats.sort_values('mean', ascending=False)

print("\nGenotype Rankings:")
for genotype, row in genotype_stats.iterrows():
    if genotype == 'Hoga':
        continue  # Skip Hoga for now, will handle separately
    print(f"{genotype:20s}: {row['mean']:7.4f} ± {row['std']:6.4f} (n={int(row['count'])})")

print("\n" + "=" * 80)
print("SPECIAL CASE: HOGA ANALYSIS")
print("=" * 80)

# Hoga per-experiment statistics
hoga_data = drought_scores_df[drought_scores_df['Genotype'] == 'Hoga']

if len(hoga_data) > 0:
    print("\nHoga by Experiment:")
    hoga_by_exp = hoga_data.groupby('Experiment')['Drought_Score'].agg(['mean', 'std', 'count'])
    for exp, row in hoga_by_exp.iterrows():
        print(f"  Experiment {int(exp)}: {row['mean']:7.4f} ± {row['std']:6.4f} (n={int(row['count'])})")
    
    print(f"\nHoga Overall:")
    hoga_mean = hoga_data['Drought_Score'].mean()
    hoga_std = hoga_data['Drought_Score'].std()
    hoga_count = len(hoga_data)
    print(f"  All Experiments: {hoga_mean:7.4f} ± {hoga_std:6.4f} (n={hoga_count})")
else:
    print("\nNo Hoga samples found in dataset")

print("\n" + "=" * 80)